<a href="https://colab.research.google.com/github/heetaamin/Assignment-4/blob/main/03_final_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# final evaluation - the held-out test set is used here for the first and only time
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import time, os, gc
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score, cohen_kappa_score,
                             matthews_corrcoef, precision_recall_fscore_support, confusion_matrix)
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTETomek

BASE = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 4/'
DATA_DIR = BASE + 'data/'
RESULTS_DIR = BASE + 'results/'
LABEL = 'label_majority'
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

vec = pd.read_csv(DATA_DIR + 'networkTraffic_vectors.csv')
dev = vec[vec['fold'] >= 0].reset_index(drop=True)
test = vec[vec['fold'] == -1].reset_index(drop=True)
print("Development:", dev.shape, " Test:", test.shape)

META_COLS = ['label_first', 'label_majority', 'is_conflict', 'n_rows', 'fold']
TREE_DROP = ['ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt']
CLUSTER_DROP = ['ct_dst_src_ltm', 'ct_src_dport_ltm', 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
KNN_DROP = TREE_DROP + CLUSTER_DROP
NOMINAL_COLS = ['proto', 'state', 'service']

# same preparation as notebooks 01 and 02, now fitted on the whole development set and applied to the test set
def prep_tree(train, val, drop_cols=TREE_DROP):
    X_train = train.drop(columns=META_COLS + drop_cols).copy()
    X_val = val.drop(columns=META_COLS + drop_cols).copy()
    proto_counts = X_train['proto'].value_counts()
    keep_protos = proto_counts[proto_counts >= 0.01 * len(X_train)].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_val['proto'] = X_val['proto'].where(X_val['proto'].isin(keep_protos), 'other')
    X_train = pd.get_dummies(X_train, columns=NOMINAL_COLS)
    X_val = pd.get_dummies(X_val, columns=NOMINAL_COLS).reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_val

def prep_knn(train, val, seed, n_features=10):
    X_train = train.drop(columns=META_COLS + KNN_DROP).copy()
    X_val = val.drop(columns=META_COLS + KNN_DROP).copy()
    X_train['service_missing'] = (X_train['service'] == '?').astype(int)
    X_val['service_missing'] = (X_val['service'] == '?').astype(int)
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
    proto_counts = X_train['proto'].value_counts()
    keep_protos = proto_counts[proto_counts >= 0.01 * len(X_train)].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_val['proto'] = X_val['proto'].where(X_val['proto'].isin(keep_protos), 'other')
    X_train = pd.get_dummies(X_train, columns=NOMINAL_COLS).astype(float)
    X_val = pd.get_dummies(X_val, columns=NOMINAL_COLS).reindex(columns=X_train.columns, fill_value=0).astype(float)
    mi = pd.Series(mutual_info_classif(X_train, train[LABEL], random_state=seed), index=X_train.columns)
    top = mi.sort_values(ascending=False).head(n_features).index.tolist()
    return X_train[top], X_val[top], mi

def oversample(X, y, mult, seed):
    counts = y.value_counts()
    target = {cls: min(n * mult, counts.max()) for cls, n in counts.items()}
    return RandomOverSampler(sampling_strategy=target, random_state=seed).fit_resample(X, y)

def smote_tomek_5x(X, y, seed):
    counts = y.value_counts()
    target = {cls: min(n * 5, counts.max()) for cls, n in counts.items()}
    smote = SMOTE(sampling_strategy=target, random_state=seed, k_neighbors=5)
    return SMOTETomek(smote=smote, random_state=seed).fit_resample(X, y)

# kappa and the Matthews correlation coefficient are added here, both are less affected by class imbalance than accuracy
def all_measures(y_true, y_pred):
    return {'macro_f1': f1_score(y_true, y_pred, average='macro'),
            'accuracy': accuracy_score(y_true, y_pred),
            'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'kappa': cohen_kappa_score(y_true, y_pred),
            'mcc': matthews_corrcoef(y_true, y_pred)}

Mounted at /content/drive
Development: (122947, 47)  Test: (30737, 47)


In [ ]:
# final models trained on the whole development set and scored on the test set
# repeated over 5 seeds so the test result has a mean and a standard deviation
# the seed changes the forest's bootstrap samples, the oversampling and the tree's tie-breaking
# results are saved after every model so a restart carries on from where it stopped
# the first seed also saves the feature importances

SEEDS = [42, 1, 2, 3, 4]
TEST_PATH = RESULTS_DIR + 'final_test_results.csv'
TEST_PRED_PATH = RESULTS_DIR + 'final_test_preds.csv'

test_results = pd.read_csv(TEST_PATH).to_dict('records') if os.path.exists(TEST_PATH) else []
done = {(r['model'], r['seed']) for r in test_results}
y_dev, y_test = dev[LABEL], test[LABEL]

def record(model_name, seed, model, X_dev_enc, X_test_enc, fit_time):
    start = time.time()
    y_pred = model.predict(X_test_enc)
    predict_time = time.time() - start
    # training score is on the original development rows, to compare with the test score for overfitting
    row = {'model': model_name, 'seed': seed, **all_measures(y_test, y_pred),
           'train_macro_f1': f1_score(y_dev, model.predict(X_dev_enc), average='macro'),
           'fit_time': fit_time, 'predict_time': predict_time,
           'predict_ms_per_instance': 1000 * predict_time / len(y_test)}
    test_results.append(row)
    pd.DataFrame(test_results).to_csv(TEST_PATH, index=False)
    preds = pd.DataFrame({'model': model_name, 'seed': seed, 'y_true': y_test.values, 'y_pred': y_pred})
    preds.to_csv(TEST_PRED_PATH, mode='a', header=not os.path.exists(TEST_PRED_PATH), index=False)
    print(f"{model_name} seed {seed}: test macro-F1 {row['macro_f1']:.4f}, train macro-F1 {row['train_macro_f1']:.4f}")

for seed in SEEDS:
    if ('tree', seed) not in done or ('forest', seed) not in done:
        X_dev_enc, X_test_enc = prep_tree(dev, test)

    # single tree: entropy, ccp_alpha=0.0003, oversampling 5x
    if ('tree', seed) not in done:
        X_res, y_res = oversample(X_dev_enc, y_dev, 5, seed)
        tree = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.0003, random_state=seed)
        start = time.time()
        tree.fit(X_res, y_res)
        fit_time = time.time() - start
        if seed == SEEDS[0]:
            pd.Series(tree.feature_importances_, index=X_dev_enc.columns,
                      name='tree_impurity').to_csv(RESULTS_DIR + 'importance_tree.csv')
        record('tree', seed, tree, X_dev_enc, X_test_enc, fit_time)

    # forest: 200 trees, half the features per split, unrestricted trees, oversampling 10x
    if ('forest', seed) not in done:
        X_res, y_res = oversample(X_dev_enc, y_dev, 10, seed)
        forest = RandomForestClassifier(n_estimators=200, max_features=0.5, criterion='entropy',
                                        n_jobs=-1, random_state=seed)
        start = time.time()
        forest.fit(X_res, y_res)
        fit_time = time.time() - start
        if seed == SEEDS[0]:
            # permutation importance: drop in test macro-F1 when one feature's values are shuffled
            perm = permutation_importance(forest, X_test_enc, y_test, scoring='f1_macro',
                                          n_repeats=3, random_state=seed, n_jobs=1)
            pd.DataFrame({'forest_impurity': forest.feature_importances_,
                          'forest_permutation': perm.importances_mean},
                         index=X_dev_enc.columns).to_csv(RESULTS_DIR + 'importance_forest.csv')
        record('forest', seed, forest, X_dev_enc, X_test_enc, fit_time)
        del forest, X_res, y_res
        gc.collect()

    # kNN: k=31, manhattan, distance weighted, top 10 features, SMOTE + Tomek
    if ('knn', seed) not in done:
        X_dev_knn, X_test_knn, mi = prep_knn(dev, test, seed)
        X_res, y_res = smote_tomek_5x(X_dev_knn, y_dev, seed)
        knn = KNeighborsClassifier(n_neighbors=31, weights='distance', metric='manhattan', n_jobs=-1)
        start = time.time()
        knn.fit(X_res, y_res)
        fit_time = time.time() - start
        if seed == SEEDS[0]:
            mi.rename('knn_mutual_info').to_csv(RESULTS_DIR + 'importance_knn.csv')
        record('knn', seed, knn, X_dev_knn, X_test_knn, fit_time)
        del knn, X_res, y_res
        gc.collect()

print("\nAll runs complete:", len(test_results), "rows")

tree seed 42: test macro-F1 0.6742, train macro-F1 0.6943
forest seed 42: test macro-F1 0.7506, train macro-F1 0.9986
knn seed 42: test macro-F1 0.6348, train macro-F1 0.8811
tree seed 1: test macro-F1 0.6800, train macro-F1 0.6920
forest seed 1: test macro-F1 0.7515, train macro-F1 0.9986
knn seed 1: test macro-F1 0.6368, train macro-F1 0.8966
tree seed 2: test macro-F1 0.6695, train macro-F1 0.6893
forest seed 2: test macro-F1 0.7469, train macro-F1 0.9986
knn seed 2: test macro-F1 0.6440, train macro-F1 0.8951
tree seed 3: test macro-F1 0.6853, train macro-F1 0.6956
forest seed 3: test macro-F1 0.7453, train macro-F1 0.9986
knn seed 3: test macro-F1 0.6395, train macro-F1 0.8955
tree seed 4: test macro-F1 0.6840, train macro-F1 0.6965
forest seed 4: test macro-F1 0.7465, train macro-F1 0.9987
knn seed 4: test macro-F1 0.6305, train macro-F1 0.8837

All runs complete: 15 rows


In [ ]:
# summary of the test results over the 5 seeds
res = pd.read_csv(TEST_PATH)
order = ['knn', 'tree', 'forest']
print(res.drop(columns='seed').groupby('model').agg(['mean', 'std']).loc[order].round(4).T)

# per-class F1, averaged over the seeds
preds = pd.read_csv(TEST_PRED_PATH)
CLASS_NAMES = ['Normal', 'Reconnaissance', 'Backdoor', 'DoS', 'Exploits', 'Analysis', 'Fuzzers', 'Worms', 'Shellcode', 'Generic']
per_class = {}
for m in order:
    f1_by_seed = []
    for seed in SEEDS:
        p = preds[(preds['model'] == m) & (preds['seed'] == seed)]
        f1_by_seed.append(f1_score(p['y_true'], p['y_pred'], average=None, labels=range(10)))
    per_class[m] = np.mean(f1_by_seed, axis=0)
per_class_df = pd.DataFrame(per_class, index=CLASS_NAMES)
per_class_df['support'] = y_test.value_counts().sort_index().values
per_class_df['forest_minus_tree'] = per_class_df['forest'] - per_class_df['tree']
per_class_df.to_csv(RESULTS_DIR + 'final_test_per_class.csv')
print()
print(per_class_df.round(4))

# precision, recall and the confusion matrix for each model, from the first seed
first = {m: preds[(preds['model'] == m) & (preds['seed'] == SEEDS[0])]['y_pred'].values for m in order}
pr_rows = []
for m in order:
    p_, r_, f_, _ = precision_recall_fscore_support(y_test, first[m], labels=range(10), zero_division=0)
    for i, name in enumerate(CLASS_NAMES):
        pr_rows.append({'model': m, 'class': name, 'precision': p_[i], 'recall': r_[i], 'f1': f_[i]})
    pd.DataFrame(confusion_matrix(y_test, first[m], labels=range(10)),
                 index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR + f'confusion_{m}.csv')
pr_df = pd.DataFrame(pr_rows)
pr_df.to_csv(RESULTS_DIR + 'final_test_precision_recall.csv', index=False)
print()
print(pr_df.pivot(index='class', columns='model', values=['precision', 'recall']).loc[CLASS_NAMES].round(3))

# how much of the error falls on the vectors that had conflicting labels
conflict = test['is_conflict'].values == 1
print(f"\nConflicting vectors in the test set: {conflict.sum()} of {len(test)} ({conflict.mean()*100:.2f}%)")
for m in order:
    wrong = first[m] != y_test.values
    print(f"{m}: {wrong.sum()} errors, {(wrong & conflict).sum()} on conflicting vectors "
          f"({(wrong & conflict).sum() / wrong.sum() * 100:.1f}% of errors), "
          f"error rate {wrong[conflict].mean()*100:.1f}% on conflicting vs {wrong[~conflict].mean()*100:.1f}% on the rest")

model                            knn     tree    forest
macro_f1                mean  0.6371   0.6786    0.7482
                        std   0.0051   0.0067    0.0027
accuracy                mean  0.8065   0.8010    0.8796
                        std   0.0016   0.0043    0.0005
weighted_f1             mean  0.8161   0.8145    0.8796
                        std   0.0013   0.0039    0.0005
balanced_accuracy       mean  0.6499   0.7159    0.7246
                        std   0.0057   0.0035    0.0025
kappa                   mean  0.7113   0.7136    0.8126
                        std   0.0020   0.0050    0.0007
mcc                     mean  0.7169   0.7319    0.8129
                        std   0.0017   0.0036    0.0007
train_macro_f1          mean  0.8904   0.6935    0.9986
                        std   0.0074   0.0029    0.0001
fit_time                mean  0.7868  10.6879  655.1177
                        std   0.1831   1.3180   24.3453
predict_time            mean  7.3618   0.0086   

In [ ]:
# which features each method finds informative
# the four columns measure different things, so the rankings are compared, not the values:
#   tree_impurity / forest_impurity - share of the total impurity reduction that came from splits on the feature
#   forest_permutation - drop in test macro-F1 when the feature's values are shuffled
#   knn_mutual_info - mutual information between the feature and the target, used to select kNN's features
imp = pd.concat([pd.read_csv(RESULTS_DIR + f'importance_{m}.csv', index_col=0) for m in ['tree', 'forest', 'knn']], axis=1)
imp.to_csv(RESULTS_DIR + 'feature_importance.csv')

top = pd.DataFrame({c: imp[c].sort_values(ascending=False).head(10).index for c in imp.columns})
top.index = range(1, 11)
print(top)
print()
print("Spearman rank correlation between the rankings:")
print(imp.corr(method='spearman').round(3))

     tree_impurity forest_impurity forest_permutation knn_mutual_info
1             sttl            sttl               sttl          sbytes
2      service_dns          sbytes             sbytes          dbytes
3           sbytes           smean              smean           smean
4       ct_srv_dst     service_dns             dbytes           dmean
5            smean          dbytes         ct_srv_dst            sttl
6           dbytes      ct_srv_dst              dmean           sload
7   ct_dst_src_ltm  ct_dst_src_ltm     ct_dst_src_ltm    ct_state_ttl
8            dmean           dmean        service_dns            dttl
9      proto_other    ct_state_ttl          service_?           dload
10    service_http          ackdat       service_http           dpkts

Spearman rank correlation between the rankings:
                    tree_impurity  forest_impurity  forest_permutation  knn_mutual_info
tree_impurity               1.000            0.789               0.643            0.678
fores

In [ ]:
# per-class results as mean and standard deviation over the five seeds,
# plus the confusion percentages and conflicting-vector error rates quoted in the report
from sklearn.metrics import f1_score, recall_score, confusion_matrix

SEEDS = [42, 1, 2, 3, 4]
CLASS_NAMES = ['Normal', 'Reconnaissance', 'Backdoor', 'DoS', 'Exploits', 'Analysis',
               'Fuzzers', 'Worms', 'Shellcode', 'Generic']
order = ['knn', 'tree', 'forest']
y_true = test[LABEL].values
conflict = test['is_conflict'].values == 1
preds = pd.read_csv(RESULTS_DIR + 'final_test_preds.csv')

def get_pred(m, seed):
    p = preds[(preds['model'] == m) & (preds['seed'] == seed)]
    return p['y_pred'].values[-len(y_true):]   # last run only, in case a seed was saved twice

f1s = {m: np.array([f1_score(y_true, get_pred(m, s), average=None, labels=range(10)) for s in SEEDS]) for m in order}
recs = {m: np.array([recall_score(y_true, get_pred(m, s), average=None, labels=range(10), zero_division=0) for s in SEEDS]) for m in order}

rows = {}
for m in order:
    rows[(m, 'f1_mean')] = f1s[m].mean(0); rows[(m, 'f1_std')] = f1s[m].std(0, ddof=1)
    rows[(m, 'recall_mean')] = recs[m].mean(0); rows[(m, 'recall_std')] = recs[m].std(0, ddof=1)
sd = pd.DataFrame(rows, index=CLASS_NAMES)
sd.to_csv(RESULTS_DIR + 'final_test_per_class_sd.csv')
print("F1 per class, mean (std) over seeds")
print(pd.DataFrame({m: [f"{a:.3f} ({b:.3f})" for a, b in zip(f1s[m].mean(0), f1s[m].std(0, ddof=1))] for m in order}, index=CLASS_NAMES))
diff = f1s['forest'] - f1s['tree']
print("\nforest minus tree, mean (std) over seeds")
print(pd.Series([f"{a:.3f} ({b:.3f})" for a, b in zip(diff.mean(0), diff.std(0, ddof=1))], index=CLASS_NAMES))
print("\nRecall per class, mean (std) over seeds")
print(pd.DataFrame({m: [f"{a:.3f} ({b:.3f})" for a, b in zip(recs[m].mean(0), recs[m].std(0, ddof=1))] for m in order}, index=CLASS_NAMES))

# row-normalised confusion percentages quoted in the text
pairs = [('tree', 'Normal', 'Fuzzers'), ('forest', 'Normal', 'Fuzzers'), ('forest', 'Fuzzers', 'Normal'),
         ('forest', 'Analysis', 'Normal'), ('tree', 'DoS', 'Exploits'), ('forest', 'DoS', 'Exploits')]
print("\nPercentage of a class predicted as another class, mean (std) over seeds")
for m, a, b in pairs:
    vals = []
    for s in SEEDS:
        cm = confusion_matrix(y_true, get_pred(m, s), labels=range(10))
        i, j = CLASS_NAMES.index(a), CLASS_NAMES.index(b)
        vals.append(100 * cm[i, j] / cm[i].sum())
    print(f"{m}: {a} as {b}: {np.mean(vals):.1f}% ({np.std(vals, ddof=1):.1f})")

# error on the conflicting vectors
print("\nConflicting vectors, mean (std) over seeds")
for m in ['tree', 'forest']:
    on_c, on_rest, share = [], [], []
    for s in SEEDS:
        wrong = get_pred(m, s) != y_true
        on_c.append(100 * wrong[conflict].mean()); on_rest.append(100 * wrong[~conflict].mean())
        share.append(100 * (wrong & conflict).sum() / wrong.sum())
    print(f"{m}: error {np.mean(on_c):.1f}% ({np.std(on_c, ddof=1):.1f}) on conflicting, "
          f"{np.mean(on_rest):.1f}% ({np.std(on_rest, ddof=1):.1f}) on the rest, "
          f"{np.mean(share):.1f}% ({np.std(share, ddof=1):.1f}) of all errors")


F1 per class, mean (std) over seeds
                          knn           tree         forest
Normal          0.879 (0.001)  0.858 (0.005)  0.930 (0.001)
Reconnaissance  0.830 (0.003)  0.884 (0.006)  0.887 (0.001)
Backdoor        0.746 (0.019)  0.649 (0.025)  0.844 (0.004)
DoS             0.470 (0.011)  0.426 (0.009)  0.576 (0.004)
Exploits        0.816 (0.002)  0.837 (0.006)  0.878 (0.001)
Analysis        0.192 (0.026)  0.303 (0.010)  0.330 (0.015)
Fuzzers         0.618 (0.002)  0.634 (0.003)  0.724 (0.002)
Worms           0.568 (0.029)  0.667 (0.019)  0.662 (0.019)
Shellcode       0.328 (0.017)  0.608 (0.018)  0.706 (0.005)
Generic         0.924 (0.002)  0.920 (0.002)  0.944 (0.001)

forest minus tree, mean (std) over seeds
Normal             0.072 (0.005)
Reconnaissance     0.004 (0.006)
Backdoor           0.196 (0.026)
DoS                0.150 (0.010)
Exploits           0.041 (0.007)
Analysis           0.027 (0.011)
Fuzzers            0.090 (0.002)
Worms             -0.005 (0.034